# 分类器拦截与计费

Claude Fable 5 在网络安全、生物学和化学等领域的高级能力带来了真实的滥用风险：让它有用的这些技能，也可能帮助不法分子构建网络攻击或危险武器。因此，Claude Fable 5 在这些特定领域内置了护栏，并且每个请求都会经过自动安全检查。这些检查会拦截以下领域的请求：

- **攻击性网络安全技术** — 制作漏洞利用代码、恶意软件或攻击工具
- **生物学与生命科学** — 实验方法或分子机制
- **开发竞争性 AI 模型** — 可能协助构建前沿 AI 系统的请求
- **提取模型的[摘要化思考](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking)**

这些护栏刻意偏保守。它们首先为稳健性而调校，这意味着正常的良性技术工作有时也会触发它们。我们在发布 Fable 5 时，对生物学和网络安全相关的所有主题都回退到 Opus 4.8，以便让你在其他所有领域更快获得 Fable 的 Mythos 级能力。上线后我们会继续降低 Fable 5 的误报率。

**API 客户应为 Claude Fable 5 到 Opus 4.8 配置回退** — 可以使用[内置的服务端回退](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#server-side-fallback)功能（在原生 Claude API 和 Claude Platform on AWS 上可用），或使用基于 Anthropic SDK 辅助函数构建的[客户端回退](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#client-side-fallback)逻辑。

在产生任何输出之前被拦截的部分请求会计费，而延续到回退请求的缓存提示词前缀按较低的缓存读取费率计费，而不是按新的缓存写入计费。**当你不使用服务端回退时**，该优惠费率需要一个回退额度令牌，SDK 的回退中间件会替你兑换该令牌 — 见[下文](#4-billing-changes)。

## 本指南涵盖的内容

1. [分类器拦截是什么样的](#1-what-a-classifier-block-looks-like)
2. [服务端回退（推荐）](#2-server-side-fallback-recommended)
3. [流式输出](#3-streaming)
4. [计费变化](#4-billing-changes)
5. [使用 SDK 实现客户端回退](#5-client-side-fallback-with-the-sdk)
6. [常见反模式](#6-common-anti-patterns)


In [ ]:
%%capture
%pip install -U "anthropic>=0.108.0"

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

PRIMARY_MODEL = "claude-fable-5"
FALLBACK_MODEL = "claude-opus-4-8"
SERVER_SIDE_FALLBACK_BETA = "server-side-fallback-2026-06-01"
FALLBACK_CREDIT_BETA = "fallback-credit-2026-06-01"

# Anthropic() reads ANTHROPIC_API_KEY from the environment. Add it to a .env
# file (loaded above) or export it in your shell before running the live examples.
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY is not set - add it to .env or export it.")

## 1. 分类器拦截是什么样的

*分类器拦截*是请求看似违反我们的护栏时 API 返回的结果。API 返回 `200`，`stop_reason` 为 `"refusal"`，并附带一个 `stop_details` 对象说明所属类别：

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "explanation": "This request triggered restrictions on violative cyber content and was blocked under Anthropic's Usage Policy..."
  },
  "content": [...]
}
```

你的逻辑分支应基于 **`stop_reason`，而不是 `content` 或 `stop_details`**。`stop_details` 仅供参考，可能为 `null`，此时应视为通用拒绝（不特定于下列类别）。

当该字段存在时，`category` 是 `"cyber"`、`"bio"`、`"frontier_llm"` 或 `"reasoning_extraction"` 之一。这有助于你细化回退选择：

| 类别 | 触发情形 ||
| --- | --- ||
| `cyber` | 攻击性网络安全内容（漏洞利用代码、恶意软件、攻击工具） ||
| `bio` | 生物学／生命科学内容（实验方法、分子机制） ||
| `frontier_llm` | 可能协助开发竞争性 AI 模型的请求 ||
| `reasoning_extraction` | 试图提取模型[摘要化思考](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking)的请求 ||

分类器拦截与**模型拒绝**（模型因其他政策原因自行拒绝）是两回事。二者都表现为 `stop_reason: "refusal"`，但 `stop_details.category` 会告诉你是哪个分类器拦截了请求。

> **说明：**当你**不**使用服务端回退功能时，`stop_details` 还会包含一个 `fallback_credit_token`，你用它把回退模型的请求按缓存读取计费 — 见[计费变化](#4-billing-changes)。


## 2. 服务端回退（推荐）

Messages API 可以替你执行回退。传入 `fallbacks`，值为 `[{"model": "claude-opus-4-8"}]`，并加上 `server-side-fallback-2026-06-01` 测试版请求头。如果 Fable 的分类器拦截了这一轮，API 会自动用 Opus 4.8 重试 — 并加以标注，让你能看出发生了什么。

自动回退功能目前支持 **Claude API** 和 **Claude Platform on AWS**。目前仅支持从 Fable 5 回退到 Opus 4.8；我们计划扩展这一范围。

```bash
curl https://api.anthropic.com/v1/messages \
  -H "x-api-key: $ANTHROPIC_API_KEY" \
  -H "anthropic-version: 2023-06-01" \
  -H "anthropic-beta: server-side-fallback-2026-06-01" \
  -H "content-type: application/json" \
  -d '{
    "model": "claude-fable-5",
    "max_tokens": 1024,
    "fallbacks": [
      { "model": "claude-opus-4-8" }
    ],
    "messages": [
      { "role": "user", "content": "Hello, world" }
    ]
  }'
```

### 回退无法执行的情形

当你已配置 `fallbacks`，但 API 无法访问回退模型 — 其速率限制已耗尽或处于过载状态 — 这一轮仍会以拒绝形式返回，而 `stop_details.recommended_model` 会给出应直接重试的规范模型 ID：

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "recommended_model": "claude-opus-4-8"
  }
}
```

`recommended_model` **仅**在这种情形下才会填充（已配置回退*且*回退无法执行）。在未配置回退的普通拦截中，该字段不存在 — 这就是[第 1 节](#1-what-a-classifier-block-looks-like)基础示例中没有它的原因。


In [ ]:
# Fable applies extra safety filters. With a fallback chain configured, the API
# retries blocked turns on the next model server-side. A stop_reason of "refusal"
# means the whole chain refused.

from anthropic import Anthropic

client = Anthropic()


def chat_turn(messages, max_tokens=1024):
    """One API call; the server handles the fallback."""
    return client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    )

### 检测回退（非流式）

回退响应在每个切换点带有一个 `{"type": "fallback"}` 内容块，而 `usage.iterations` 记录各模型的用量。注意，**粘性服务**的一轮 — 因为对话中较早的某一轮发生了回退而直接路由到回退模型 — 并*不*带回退块，因为请求被直接路由，`content` 中没有可标记的边界。`usage.iterations` 是判断某一轮是否由回退模型服务的可靠方式。


In [ ]:
def fallback_hops(response):
    """(from_model, to_model) for each hop that ran and blocked this turn."""
    hops = []
    for b in response.content:
        if getattr(b, "type", None) == "fallback":
            d = b.model_dump() if hasattr(b, "model_dump") else dict(b)
            hops.append((d["from"]["model"], d["to"]["model"]))
    return hops


def served_by_fallback(response):
    """True whenever a fallback model served the response, INCLUDING a
    sticky-served turn (which carries no fallback block). usage.iterations is
    the best way to check whether a turn was served by a fallback model."""
    iters = getattr(response.usage, "iterations", None) or []
    return any(
        (i.get("type") if isinstance(i, dict) else getattr(i, "type", None)) == "fallback_message"
        for i in iters
    )


response = chat_turn([{"role": "user", "content": "Hello, world"}])
hops = fallback_hops(response)
for from_model, to_model in hops:
    print(f"[{from_model} blocked \u2014 continued on {to_model}]")
if not hops and served_by_fallback(response):
    print(f"[sticky: served directly by {response.model}]")

### 在流式输出中检测回退

留意 `content_block_start` 事件中内容为 `{"type": "fallback"}` 的块 — 它标记流内的切换点。但要获得每一轮的确定答案，请在**最终**消息上检查 `usage.iterations`，与非流式响应的做法完全一致。这一检查在任何情况下都可靠，包括直接由回退模型服务的流，因此应把它作为权威依据，而不是仅依赖流内事件。


In [ ]:
with client.beta.messages.stream(
    model=PRIMARY_MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, world"}],
    betas=[SERVER_SIDE_FALLBACK_BETA],
    fallbacks=[{"model": FALLBACK_MODEL}],
) as stream:
    for event in stream:
        if (
            getattr(event, "type", None) == "content_block_start"
            and getattr(event.content_block, "type", None) == "fallback"
        ):
            fb = event.content_block
            fb = fb.model_dump() if hasattr(fb, "model_dump") else dict(fb)
            print(f"[switching: {fb['from']['model']} -> {fb['to']['model']}]")
    final = stream.get_final_message()

# Definitive per-turn check, same as non-streaming: usage.iterations also
# catches a stream served directly by the fallback model (no in-stream event).
if served_by_fallback(final):
    print(f"[fallback model served this stream: {final.model}]")

### 回退响应的结构

回退响应包含 `message.model`（最终作答的模型）、一个标记每个切换点的 `{"type": "fallback"}` 内容块，以及 `usage.iterations` 中按次统计的用量：

```json
{
  "id": "msg_01Ab...",
  "type": "message",
  "role": "assistant",
  "model": "claude-opus-4-8",
  "content": [
    { "type": "fallback", "from": { "model": "claude-fable-5" }, "to": { "model": "claude-opus-4-8" } },
    { "type": "text", "text": "..." }
  ],
  "stop_reason": "end_turn",
  "stop_details": null,
  "usage": {
    "input_tokens": 412, "output_tokens": 264,
    "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0,
    "iterations": [
      { "type": "message", "model": "claude-fable-5", "input_tokens": 408, "output_tokens": 0,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 },
      { "type": "fallback_message", "model": "claude-opus-4-8", "input_tokens": 412, "output_tokens": 264,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 }
    ]
  }
}
```

**按次覆盖项。**每个回退条目可以仅针对该次尝试覆盖 `max_tokens`、`thinking`、`output_config` 和 `speed`（`output_config` 和 `speed` 还需要与相应顶层字段相同的测试版请求头）。合并某条目覆盖项后的请求，必须是针对该条目模型、格式正确的直接请求。

```json
{
  "model": "claude-fable-5",
  "max_tokens": 1024,
  "fallbacks": [
    { "model": "claude-opus-4-8", "max_tokens": 8192, "thinking": {"type": "disabled"}, "speed": "fast" }
  ],
  "messages": [
    { "role": "user", "content": "Hello, world" }
  ]
}
```

**计费。**`usage.input_tokens` 每一轮只计一次。`usage.output_tokens` 反映作答内容。如果需要按模型精确分摊，请使用 `usage.iterations`。


## 3. 流式输出

在流式输出中，回退被设计为自动生效。如果分类器在**任何输出到达你之前**就拦截，流会从回退模型的响应开始。这次重试不可见，也不会发出回退 SSE 事件。

如果分类器在**流中途**拦截，重试也在同一条流上进行：已有的部分输出被保留，一个 `{"type": "fallback"}` 内容块标记边界，回退模型从这部分输出继续。已流出的内容不会被丢弃。


In [ ]:
def stream_turn(messages, max_tokens=1024):
    with client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    ) as stream:
        # Nothing streamed is ever discarded: after a mid-stream block, the
        # final message is partial + fallback block + continuation.
        final = stream.get_final_message()

    if final.stop_reason == "refusal":
        return final  # the whole chain refused
    text = "".join(b.text for b in final.content if b.type == "text")
    print(f"{final.model}: {text}")
    return final

## 4. 计费变化

本节说明被拦截的请求和回退如何计费。使用回退和 Anthropic SDK 辅助函数时，这些行为会自动生效。**只有采用缓存读取计费这一变化需要采取行动，且仅当你不使用服务端回退时。**

**1. 流前拦截在部分类别中会计费。**在返回任何输出词元之前就被拦截的请求，如果其 `stop_details.category` 为 `"bio"`、`"frontier_llm"` 或 `"reasoning_extraction"`，则按所请求模型的费率正常计费。截至 2026 年 9 月 23 日，其他任何类别或无类别的流前拦截不计费；[拒绝与回退](https://platform.claude.com/docs/en/build-with-claude/refusals-and-fallback#how-refusals-are-billed)列有当前清单。计费仅适用于护栏误报率较低的类别，因为计费主要提高反复试图绕过护栏的成本。如果在计费的拦截之后进行回退，回退请求同样会计费，因此两者都收费，不过下面第 2 项会抵消重试的提示词缓存成本。

在流中途被拦截的请求（输出词元已开始流式传输之后），无论属于哪个类别，都按正常费率对输入词元和已流出的输出计费。

**2. Fable 5 → Opus 4.8 回退的输入词元按缓存命中计费。**通常，切换到另一个模型按[缓存*写入*](https://platform.claude.com/docs/en/build-with-claude/prompt-caching#how-prompt-caching-works)计费，即比基础输入词元价格高 1.25 倍（5 分钟 TTL）或 2 倍（60 分钟 TTL）。而我们对这些 Opus 词元按已经缓存过的情况计费，即按缓存*读取*计费，为基础输入词元价格的 10%。

- **使用服务端回退功能：**该计费变化自动生效。
- **不使用服务端回退功能：**见下面的额度令牌流程。

### 兑换回退额度令牌（仅限客户端回退）

被安全分类器拦截的 Fable 请求会在 `stop_details` 中包含一个 `fallback_credit_token`。该令牌**仅**在被拦截请求存在可计费的缓存前缀时出现，否则为 `null`。

兑换方法：

1. 发送后续的 Opus 4.8 请求时带上 `anthropic-beta: fallback-credit-2026-06-01` 请求头。
2. 把该令牌作为顶层 `fallback_credit_token` 参数传入。
3. 保持提示词构成字段与被拦截请求**完全一致** — 即完全相同的 `system`、`messages` 和 `tools`。

这样，在 Fable 请求上缓存的前缀就按缓存读取费率而非缓存写入费率计费。切换成本被退还，因此重试的费用与对话一直使用 Opus 时的费用相同。

**有效期：**该令牌仅在被拦截的 Fable 5 请求后 **5 分钟内**发出、且来自**同一组织和同一工作区**的 Opus 4.8 请求上有效。

**流中途拦截。**如果 Fable 5 请求在流式输出词元的过程中被拦截，`stop_details` 会在额度令牌之外包含 `fallback_has_prefill_claim: true`。这意味着在后续的 Opus 4.8 请求中，你可以把那段部分输出作为 assistant 前导填充追加，并从 Fable 停止的位置继续 — 这在 Opus 4.8 请求中[通常是不允许的](https://platform.claude.com/docs/en/test-and-evaluate/strengthen-guardrails/increase-consistency#prefill-claudes-response)。


In [ ]:
def redeem_credit_after_block(blocked_response, messages, system=None, tools=None, max_tokens=1024):
    """Retry a classifier-blocked Fable turn on Opus 4.8, redeeming the
    fallback credit token so the cached prefix is billed at the cache-read
    rate. Use this only when you are NOT using server-side fallback.

    Pass the same `system`, `messages`, and `tools` as the blocked request."""
    details = blocked_response.stop_details
    credit = getattr(details, "fallback_credit_token", None) if details else None

    # The system, messages, and tools must be IDENTICAL to the blocked request.
    kwargs = {"model": FALLBACK_MODEL, "max_tokens": max_tokens, "messages": messages}
    if system is not None:
        kwargs["system"] = system
    if tools is not None:
        kwargs["tools"] = tools
    if credit is not None:  # present only when the blocked request had a cached prefix
        kwargs["betas"] = [FALLBACK_CREDIT_BETA]
        kwargs["extra_body"] = {"fallback_credit_token": credit}

    return client.beta.messages.create(**kwargs)

## 5. 使用 SDK 实现客户端回退

服务端回退在原生 Claude API 和 Claude Platform on AWS 上可用，但目前不支持 Amazon Bedrock、Vertex AI、Microsoft Foundry 或 Message Batches API。对于这些平台，或者任何时候你想把回退逻辑放在客户端，Anthropic SDK（Python、TypeScript、Go、Java、C#）都提供了**拒绝回退中间件**。

在客户端上一次性配置回退模型列表和一个 `BetaFallbackState`，之后照常调用 `client.beta.messages`。该中间件会：

- 对 `stop_reason: "refusal"` 的那一轮，用列表中的下一个模型重试（如果回退模型也拒绝，则继续沿链条向下；如果每个条目都拒绝，则返回原始拒绝结果而不抛出异常）；
- **在每个请求上自动发送 `fallback-credit-2026-06-01` 测试版请求头**，因此你无需自行管理令牌即可获得[第 4 节](#4-billing-changes)的缓存读取计费变化；
- 替你管理对话历史中的 `fallback` 内容块；
- 在 `BetaFallbackState` 中记录接受请求的模型，使后续轮次固定使用该模型。

它与服务端的 `fallbacks` 参数**互斥**，二者只能选其一。（如果在使用该中间件的应用中要发送服务端 `fallbacks` 请求，请使用一个不装中间件的独立客户端实例。）


In [ ]:
from anthropic import Anthropic, BetaFallbackState, BetaRefusalFallbackMiddleware

# Install the middleware once, with your fallback chain. No per-request betas needed.
client = Anthropic(
    middleware=[BetaRefusalFallbackMiddleware([{"model": FALLBACK_MODEL}])],
)

state = BetaFallbackState()  # reuse across turns to pin follow-ups to the accepting model

# Non-streaming: a refused Fable turn is retried on Opus 4.8 transparently.
with state:
    message = client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    )
print(f"served by: {message.model}")

# Streaming: on a refusal the middleware splices the fallback model's events
# onto the same open stream.
with (
    state,
    client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    ) as stream,
):
    for event in stream:
        if event.type == "text":
            print(event.text, end="", flush=True)
    final = stream.get_final_message()
print(f"\nserved by: {final.model}")

## 6. 常见反模式

**在每次请求上设置回退，而不是每个账号设置一次。**不存在账号级或会话级的开关来启用 Opus 4.8 回退。每次 API 调用都必须包含回退配置。未启用回退的调用会返回拒绝，而不会悄悄用回退模型重试。

**审查每一条构建请求的代码路径。**重试按钮、消息重新生成、工具使用续写等功能常常自行构建请求，每一条都可能悄悄漏掉回退配置。请在每个入口点显式设置回退。（参见[迁移指南](https://platform.claude.com/docs/en/about-claude/models/migration-guide)和 [claude-api skill](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/claude-api-skill)，以便在代码库中快速添加回退。）

**在子智能体调用中包含回退，并预期按智能体分别生效。**如果你的应用在一个会话中运行多个智能体，每个智能体的调用都需要回退配置。发生拒绝时，只有收到拒绝的那个智能体会切换到回退模型 — 其他智能体仍留在 Fable 上。不要以为某个智能体的回退适用于整个会话。（Claude Code 中的子智能体也是如此：只有遇到拒绝的子智能体会回退到 Opus；会话其余部分继续在 Fable 5 上运行。）

**拒绝后重新提交同一请求只会再次被拒绝。**被拒绝的内容仍在对话历史中，因此向同一模型重新提交会再次触发拦截。遇到 `stop_reason: "refusal"` 时，应在回退模型上重试，并设置一个独立标志，让你的路由在会话剩余时间里保持在回退模型上。

**基于 `usage.iterations` 构建实际服务模型的分析，而不是基于你所请求的模型。**响应中的 `model` 字段是实际作答的模型，因此由回退模型服务的那一轮会报告 Opus 4.8。按*请求的模型*记录的分析数据在使用回退时总是错的。可靠的逐轮检查是最终用量记录中的 `usage.iterations`。

**谨慎处理流式输出的截断。**当你在流中途遇到分类器拦截时，省略出现在回退之前的任何 `thinking`、`tool_use` 或其他块。被截断的 `tool_use` 块是无法解析的 JSON，而另一个模型的 `thinking` 块会导致下一次调用出错。如果不使用服务端回退而要继续，请使用 `stop_details` 中的 `fallback_has_prefill_claim` 授权，而不是把部分响应粘贴进一个已完成的 assistant 轮次。
